# PART 1. Import and Clean Wind Data

- AZMet Data: az06, az05, az22, az26, az19, az15; 2005-01-01 through 2019-12-31
- ERA5 Data: u/v wind components at 10 m and 100 m
- Goal: Download data, clean data, save data as tables for use in training or generation
- `wind_pipeline.py` stores all called functions
- `%pip install -r requirements-wind.txt` for all dependancies

References: [AZMet format](https://azmet.arizona.edu/azmet/raw2003.htm),
[AZMet time and missing values](https://azmet.arizona.edu/azmet/data.html),
[ERA5 dataset](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries).

In [20]:
"""------------------------ LIBRARY IMPORTS AND SETUP ------------------------"""

from pathlib import Path
import importlib.metadata
import json
import os
import pandas as pd
from IPython.display import display
import wind_pipeline as wp

ROOT = Path.cwd()
RAW = ROOT / 'downloaded_datasets/wind_only/raw'
OUT = ROOT / 'analyses_output_files/wind_pipeline'
SITES, START, END = wp.SITES, wp.START, wp.END

#------ alter here ------
REFERENCE = 'az06'
TRAIN_SITES, VALIDATION_SITES, TEST_SITES = ['az05', 'az22', 'az26'], ['az19'], ['az15']
#------ alter here ------

FIT_END_LOCAL, MIN_COVERAGE = '2015-12-31', 0.90
DOWNLOAD_AZMET = True   # just turn off if we have raw data already and want to analyse it only
DOWNLOAD_ERA5 = True    # just turn off if we have raw data already and want to analyse it only
PREPARE_MODEL_INPUTS = False
for name in ['observations', 'metadata', 'quality', 'hidden_evaluation']:
    (OUT / name).mkdir(parents=True, exist_ok=True)
assert set([REFERENCE, *TRAIN_SITES, *VALIDATION_SITES, *TEST_SITES]) == set(SITES)
wp.save_json(OUT / 'metadata/site_split.json', {
    'reference': REFERENCE, 'train': TRAIN_SITES, 'validation': VALIDATION_SITES,
    'test': TEST_SITES, 'fit_end_local': FIT_END_LOCAL})

## just checking if cds is working, no calls yet
import cdsapi
print('cdsapi version:', importlib.metadata.version('cdsapi'))
print('Local credentials configured:', (Path.home() / '.cdsapirc').is_file() or bool(os.getenv('CDSAPI_KEY')))
verification = OUT / 'metadata/cdsapi_verification.json'
if verification.exists():
    display(json.loads(verification.read_text(encoding='utf-8')))

cdsapi version: 0.7.7
Local credentials configured: True


{'package_version': '0.7.7',
 'client_initialization': 'passed',
 'connection': 'passed',
 'sample_download': 'blocked',
 'http_status': 403,
 'reason': 'required licences not accepted',
 'licence_url': 'https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=download#manage-licences'}

In [21]:
"""------------------------ DOWNLOAD AZMET FILES ------------------------"""
# won't make a new call if chached files are here already

manifest, station_notes, source_issues = wp.cached_azmet(RAW, download=DOWNLOAD_AZMET)
display(manifest.groupby('site_id').agg(files=('year', 'size'), first_year=('year', 'min'), last_year=('year', 'max')))
if not source_issues.empty:
    display(source_issues.groupby('kind').size().rename('unavailable_sources'))

,files,first_year,last_year
site_id,,,
az05,15,2005,2019
az06,15,2005,2019
az15,15,2005,2019
az19,15,2005,2019
az22,15,2005,2019
az26,15,2005,2019


kind
notes_http    82
Name: unavailable_sources, dtype: int64

In [22]:
"""------------------------ METADATA AND MEASUREMENT HEIGHT ------------------------"""
# assuming 3m height for all time (based on current height)

metadata = wp.load_metadata(RAW)
metadata['measurement_height_m'] = 3.0
events = wp.documented_station_events(station_notes)
for site in SITES:
    if not events.empty:
        history = ' '.join(events.loc[events.site_id.eq(site), 'description'])
        metadata.loc[metadata.site_id.eq(site), 'station_history'] += ' ' + history
wp.save_tables(OUT / 'metadata', {'documented_station_events': events})
display(metadata[['site_id', 'latitude_deg', 'longitude_deg', 'ground_elevation_m', 'measurement_height_m']])
display(events)

,site_id,latitude_deg,longitude_deg,ground_elevation_m,measurement_height_m
0,az06,33.068941,-111.972244,362.0,3.0
1,az05,32.979989,-111.605677,423.0,3.0
2,az22,33.193943,-111.532582,462.0,3.0
3,az26,33.409193,-112.677836,301.0,3.0
4,az19,32.927149,-112.896735,221.0,3.0
5,az15,33.478443,-112.098046,334.0,3.0


,site_id,start_local,end_local,kind,date_precision,first_hour,last_hour,source,description
0,az06,2008-04-16,2008-04-16,hourly_outage,hour,1.0,16.0,https://azmet.arizona.edu/azmet/data/0608note.txt,Battery failure; hourly hours 1-16 lost; only ...
1,az22,2006-01-01,2006-12-31,equipment_change_date_unknown,year,NaN,NaN,https://azmet.arizona.edu/azmet/data/2206note.txt,Power faults and equipment/sensor replacement ...
2,az15,2009-10-04,2009-10-05,substituted_wind,day,NaN,NaN,https://azmet.arizona.edu/azmet/data/1509note.txt,Mast fell; both days wind replaced with Phoeni...


In [23]:
"""------------------------ CLEAN HOURLY AZMET WIND DATA ------------------------"""
# read csv, retain only: 1 (year), 2 (date), 3 (hour), 11 (scalar speed), 12 (vector speed), 13 (direction)
# convert all to UTC time
# invalidate negative speeds, malformed rows, and out-of-range directions
# mask zero-vector directions, remove duplicate timestamps
# create audit that has original fields and some other metadata for quality analytics

wind, wind_audit, gaps = wp.clean_azmet(manifest, metadata, START, END, station_notes)
assert wind.columns.tolist() == wp.WIND_COLUMNS
assert not wind.duplicated(['site_id', 'time_utc']).any()
wp.save_tables(OUT / 'quality', {'wind_audit': wind_audit, 'gaps': gaps,
                               'quality_summary': wp.quality_summary(wind_audit).reset_index()})
display(wind.head())

,site_id,time_utc,height_m,speed_mps,direction_deg,vector_speed_mps,speed_valid,direction_valid
0,az05,2005-01-01 08:00:00+00:00,3.0,1.5,106.0,1.4,True,True
1,az05,2005-01-01 09:00:00+00:00,3.0,0.7,182.0,0.5,True,True
2,az05,2005-01-01 10:00:00+00:00,3.0,0.7,236.0,0.2,True,True
3,az05,2005-01-01 11:00:00+00:00,3.0,0.8,112.0,0.8,True,True
4,az05,2005-01-01 12:00:00+00:00,3.0,0.7,119.0,0.6,True,True


In [24]:
"""------------------------ VERIFY COMMON TIME PERIOD ------------------------"""
# just checking to make sure the 15-years coverage of data is present for all sites

coverage_station, coverage_year, coverage_month = wp.coverage_report(wind, SITES, START, END)
wp.save_tables(OUT / 'quality', {'coverage_station': coverage_station,
    'coverage_year': coverage_year, 'coverage_month': coverage_month}, csv=True)
display(coverage_station)
COMMON_PERIOD = wp.select_common_period(coverage_month, [REFERENCE, *TRAIN_SITES], MIN_COVERAGE)
if COMMON_PERIOD is None:
    raise ValueError('No period meets the declared coverage threshold.')
COMMON_START, COMMON_END = COMMON_PERIOD
wp.save_json(OUT / 'metadata/common_period.json', {
    'period_local': list(COMMON_PERIOD), 'selection_sites': [REFERENCE, *TRAIN_SITES],
    'minimum_monthly_speed_coverage': MIN_COVERAGE, 'era5_joint_availability_checked_separately': True})
print('AZMet common period:', COMMON_PERIOD)

,site_id,expected,present,speed_valid,direction_valid,speed_coverage,direction_coverage,missing_hours
0,az05,131472,131424,131424,131124,0.999635,0.997353,48
1,az06,131472,131376,131360,131147,0.999148,0.997528,96
2,az15,131472,131376,131328,129688,0.998905,0.986431,96
3,az19,131472,131448,131448,131237,0.999817,0.998213,24
4,az22,131472,131352,131352,131192,0.999087,0.997870,120
5,az26,131472,131400,131400,131241,0.999452,0.998243,72


AZMet common period: ('2005-01-01', '2019-12-31')


In [25]:
"""------------------------ DOWNLOAD AND MATCH ERA5 FILES ------------------------"""

## download the files, or retrieve from cache
ERA_RAW = RAW / 'era5'
if DOWNLOAD_ERA5:
    era_paths = wp.acquire_era5(metadata, ERA_RAW, START, END)
else:
    era_paths = {site: ERA_RAW / f'{site}_{START}_{END}.download' for site in SITES}
    era_paths = {site: path for site, path in era_paths.items() if path.exists()}
print('Cached ERA5 stations:', sorted(era_paths))

## load, inspect names and units, pair with AZMET data
era5, era5_grid, metadata, inspection = wp.load_era5_collection(era_paths, metadata)
era5_coverage = wp.era5_coverage_report(era5, SITES, START, END)
wp.save_tables(OUT / 'quality', {'era5_coverage_month': era5_coverage}, csv=True)
wp.save_json(OUT / 'metadata/era5_netcdf_inspection.json', inspection)
ERA5_COMPLETE = set(era_paths) == set(SITES) and era5_coverage.era5_valid.eq(era5_coverage.expected).all()
display(era5_coverage.groupby('site_id')[['expected', 'era5_valid']].sum())
if inspection:
    display(pd.DataFrame(inspection)[['site_id', 'dimensions', 'variables']])
print('ERA5 coverage complete:', ERA5_COMPLETE)

2026-11-16 12:36:41,070 INFO [2026-10-06T00:00:00Z] **Data Stores Service protection measures** affecting data downloads and account access: [please read our Announcement](https://forum.ecmwf.int/t/data-stores-service-protection-measures-affecting-data-downloads-and-account-access-cds-ads-ewds-ecds/15481) before enquiring with ECMWF Support.


Cached ERA5 stations: ['az05', 'az06', 'az15', 'az19', 'az22', 'az26']


,expected,era5_valid
site_id,,
az05,131472,131472
az06,131472,131472
az15,131472,131472
az19,131472,131472
az22,131472,131472
az26,131472,131472


,site_id,dimensions,variables
0,az06,{'valid_time': 131496},"{'u100': {'GRIB_NV': 0, 'GRIB_Nx': 1440, 'GRIB..."
1,az05,{'valid_time': 131496},"{'u100': {'GRIB_NV': 0, 'GRIB_Nx': 1440, 'GRIB..."
2,az22,{'valid_time': 131496},"{'u100': {'GRIB_NV': 0, 'GRIB_Nx': 1440, 'GRIB..."
3,az26,{'valid_time': 131496},"{'u100': {'GRIB_NV': 0, 'GRIB_Nx': 1440, 'GRIB..."
4,az19,{'valid_time': 131496},"{'u100': {'GRIB_NV': 0, 'GRIB_Nx': 1440, 'GRIB..."
5,az15,{'valid_time': 131496},"{'u100': {'GRIB_NV': 0, 'GRIB_Nx': 1440, 'GRIB..."


ERA5 coverage complete: True


In [26]:
"""------------------------ SAVE CLEANED DATA ------------------------"""

wp.save_tables(OUT / 'observations', {'wind': wind, 'era5': era5})
wp.save_tables(OUT / 'metadata', {'metadata': metadata, 'era5_grid_coordinates': era5_grid})
metadata.to_json(OUT / 'metadata/metadata.json', orient='records', indent=2)
wp.save_tables(OUT / 'hidden_evaluation', {'test_observations': wind.loc[wind.site_id.isin(TEST_SITES)]})
status = {'era5_complete': bool(ERA5_COMPLETE), 'azmet_originals': len(manifest),
    'common_period_local': list(COMMON_PERIOD), 'historical_heights_verified': bool(metadata.measurement_height_m.notna().all()),
    'model_inputs_requested': PREPARE_MODEL_INPUTS,
    'status': 'Quality checks complete; review masks before use' if ERA5_COMPLETE else 'ERA5 pending: accept licence, enable download, rerun'}
wp.save_json(OUT / 'metadata/pipeline_status.json', status)
display(status)

{'era5_complete': True,
 'azmet_originals': 90,
 'common_period_local': ['2005-01-01', '2019-12-31'],
 'historical_heights_verified': True,
 'model_inputs_requested': False,
 'status': 'Quality checks complete; review masks before use'}

In [27]:
"""------------------------ MODEL INPUT PREP - ATTENTION TRANSFORMER OPTION ------------------------"""
# this can be skipped by setting PREPARE_MODEL_INPUTS = False in setup code block


if PREPARE_MODEL_INPUTS:
    if not ERA5_COMPLETE:
        raise ValueError('Complete ERA5 retrieval and quality checks before preparing model inputs.')
    prepared = wp.prepare_model_inputs(
        wind.loc[wind.site_id.eq(REFERENCE)], wind.loc[wind.site_id.isin(TRAIN_SITES)],
        wind.loc[wind.site_id.isin(VALIDATION_SITES)], era5, metadata,
        REFERENCE, TRAIN_SITES, VALIDATION_SITES, TEST_SITES,
        COMMON_START, COMMON_END, FIT_END_LOCAL, OUT)
    display(prepared)
else:
    print('Optional model-input preparation skipped.')

Optional model-input preparation skipped.
